In [109]:
import os
import re
import pandas as pd
from openpyxl import load_workbook

# Normalization campaign names
def normalize_campaign_name(name):
    """Normalize campaign names for comparison."""
    if not name:
        return ""
    
    # Replace slashes with underscores and hyphens with underscores for consistency
    name = name.replace('/', '_').replace('-', '_')
    
    # Replace apostrophes with underscores
    name = name.replace("'", "_")
    
    # Normalize dates (e.g., 2-29-24 -> 2/29/24)
    name = re.sub(r'(\d{1,2})-(\d{1,2})-(\d{2,4})', r'\1/\2/\3', name)
    
    # Replace multiple underscores with a single underscore
    name = re.sub(r'_+', '_', name)
    
    # Remove trailing underscores
    name = name.strip('_')
    
    # Convert to lowercase
    return name.lower()

# Returns Excel Row of matched Campaign
def find_matching_campaign(normalized_folder, df):
    for _, row in df.iterrows():
        normalized_excel = row['Normalized Campaign Name']
        if normalized_folder == normalized_excel or normalized_folder in normalized_excel or normalized_excel in normalized_folder:
            return row.to_dict()
    return None

# Function to verify and display mappings
def verify_folders_against_excel(folder_path, excel_file, sheet_name, column, start_row):
    """Verify that every normalized folder name is listed in a normalized Excel column."""
    # Get folder names and normalize them
    folder_names = {name: normalize_campaign_name(name) for name in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, name))}
    
    # Load Excel data
    workbook = load_workbook(excel_file, data_only=True)
    sheet = workbook[sheet_name]
    
    # Extract and normalize campaign names from Excel
    df = pd.DataFrame(sheet.iter_rows(min_row=start_row, min_col=column, max_col=column, values_only=True), columns=['Original Campaign Name'])
    df['Normalized Campaign Name'] = df['Original Campaign Name'].apply(lambda x: normalize_campaign_name(x) if x else '')
    
    # Compare folders with Excel data
    matched_folders = {}
    unmatched_folders = []
    for folder, normalized_folder in folder_names.items():
        match, normalized_excel = find_matching_campaign(normalized_folder, df)
        if match:
            matched_folders[folder] = match
        else:
            unmatched_folders.append((folder, normalized_folder, normalized_excel))
    
    # Display results
    print("\nMatched Folders:")
    for folder, campaign_name in matched_folders.items():
        print(f"  Folder: {folder} -> Excel Match: {campaign_name}")
    
    print("\nUnmatched Folders:")
    if unmatched_folders:
        for folder, normalized_folder, normalized_excel in unmatched_folders:
            print(f"  Folder: {folder}")
            print(f"    Normalized Folder: {normalized_folder}")
            print(f"    Normalized Excel: {normalized_excel}")
    else:
        print("  All folders were matched.")
    
    # Summary
    print(f"\nSummary:")
    print(f"  Total Folders: {len(folder_names)}")
    print(f"  Correctly Matched Folders: {len(matched_folders)}")
    print(f"  Unmatched Folders: {len(unmatched_folders)}")

# Paths and configurations
folder_path = r"GenAI - Uber Creatives/GenAI - Uber Creatives Old"
excel_file = r"GenAI - Uber Creatives/Creatives.xlsx"
sheet_name = "Data"
column = 2  # Column B
start_row = 2  # Start from row 2

# Run the verification
verify_folders_against_excel(folder_path, excel_file, sheet_name, column, start_row)



Matched Folders:
  Folder: Aleph_Flash_Fifty_11-24-11-25 -> Excel Match: Original Campaign Name
  Folder: Aleph_H&M_10-20 - 10-29 -> Excel Match: Original Campaign Name
  Folder: Aleph_H&M_10-30 - 11-12 -> Excel Match: Original Campaign Name
  Folder: CPG_Aperol_Spritz_MX_6-19-24-6-21-24_JourneyAd_Video XP -> Excel Match: Original Campaign Name
  Folder: CPG_Arcor_CL_JUNIO_06-01-06-30_JourneyAd -> Excel Match: Original Campaign Name
  Folder: CPG_Campari_US_RootsPicnic_5-30-24-6-2-24_JourneyAd -> Excel Match: Original Campaign Name
  Folder: CPG_Campari_US_Wildturkey_6-6-24-6-10-24_JourneyAd -> Excel Match: Original Campaign Name
  Folder: CPG_CCSA_MX_Journey_06-18_06-30_Meals_Q2 -> Excel Match: Original Campaign Name
  Folder: CPG_Coca-Cola_US_MoodFood_11-15-23_12-17-23_JourneyAd -> Excel Match: Original Campaign Name
  Folder: CPG_Cocacola_US_Moodfood_17-6-24_28-7-24_JourneyAd_Video XP -> Excel Match: Original Campaign Name
  Folder: CPG_CocaCola_US_SimplyEssenceFest_7-03-24-07-08-2

In [111]:
def verify_folders_against_excel(folder_path, excel_file, sheet_name, column, start_row):
    """Verify folder names against Excel and list Excel entries not found in folders."""
    # Get folder names and normalize them
    folder_names = {normalize_campaign_name(name): name for name in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, name))}
    
    # Load Excel data
    workbook = load_workbook(excel_file, data_only=True)
    sheet = workbook[sheet_name]
    
    # Extract and normalize campaign names from Excel
    df = pd.DataFrame(sheet.iter_rows(min_row=start_row, min_col=column, max_col=column, values_only=True), columns=['Original Campaign Name'])
    df['Normalized Campaign Name'] = df['Original Campaign Name'].apply(lambda x: normalize_campaign_name(x) if x else '')
    
    # Compare folders with Excel data
    matched_folders = {}
    unmatched_folders = []
    excel_not_in_folders = []
    
    for _, row in df.iterrows():
        normalized_excel = row['Normalized Campaign Name']
        original_excel = row['Original Campaign Name']
        
        if normalized_excel in folder_names:
            matched_folders[folder_names[normalized_excel]] = original_excel
        else:
            excel_not_in_folders.append(original_excel)
    
    for normalized_folder, original_folder in folder_names.items():
        if original_folder not in matched_folders:
            unmatched_folders.append((original_folder, normalized_folder))
    
    # Display results
    print("\nMatched Folders:")
    for folder, campaign_name in matched_folders.items():
        print(f"  Folder: {folder} -> Excel Match: {campaign_name}")
    
    print("\nUnmatched Folders (present in directory but not in Excel):")
    if unmatched_folders:
        for folder, normalized_folder in unmatched_folders:
            print(f"  Folder: {folder}")
            print(f"    Normalized Folder: {normalized_folder}")
    else:
        print("  All folders were matched.")
    
    print("\nExcel Entries Not Found in Folders:")
    if excel_not_in_folders:
        for entry in excel_not_in_folders:
            print(f"  {entry}")
    else:
        print("  All Excel entries were found in folders.")
    
    # Summary
    print(f"\nSummary:")
    print(f"  Total Folders: {len(folder_names)}")
    print(f"  Total Excel Entries: {len(df)}")
    print(f"  Correctly Matched Folders: {len(matched_folders)}")
    print(f"  Unmatched Folders: {len(unmatched_folders)}")
    print(f"  Excel Entries Not Found in Folders: {len(excel_not_in_folders)}")

# Paths and configurations
folder_path = r"GenAI - Uber Creatives/GenAI - Uber Creatives - NEW"
excel_file = r"GenAI - Uber Creatives/Creatives.xlsx"
sheet_name = "Data"
column = 2  # Column B
start_row = 2  # Start from row 2

verify_folders_against_excel(folder_path, excel_file, sheet_name, column, start_row)


Matched Folders:
  Folder: SVT_Super.com_US_Pilot_5-3-24-5-31-24_JourneyAd -> Excel Match: SVT_Super.com_US_Pilot_5/3/24-5/31/24_JourneyAd
  Folder: SVT_FanDuel_US_Derby_5-1-24-5-4-24_JourneyAd -> Excel Match: SVT_FanDuel_US_Derby_5/1/24-5/4/24_JourneyAd
  Folder: ENT_McDonalds_SA_12-06_12-31_JourneyAD -> Excel Match: ENT_McDonalds_SA_12/06_12/31_JourneyAD
  Folder: CPG_Labatt_CA_Journey_6-15-6-16_John_Summit -> Excel Match: CPG_Labatt_CA_Journey_6/15-6/16_John_Summit
  Folder: SVT_Apple_US_ApplePayIdentity_05-06-24-07-31-24_JourneyAd -> Excel Match: SVT_Apple_US_ApplePayIdentity_05/06/24-07/31/24_JourneyAd
  Folder: SVT_Anduril_US_CACampaign_11-27-23-12-03-23_JourneyAd -> Excel Match: SVT_Anduril_US_CACampaign_11/27/23-12/03/23_JourneyAd
  Folder: SVT_Deezer_FR_Concerts_2-29 -> Excel Match: SVT_Deezer_FR_Concerts_2/29
  Folder: SVT_VH1_US_Celebritysquares_10-30-23_11-17-23_JourneyAd -> Excel Match: SVT_VH1_US_Celebritysquares_10/30/23_11/17/23_JourneyAd
  Folder: CPG_Unilever_US_Dove

In [30]:
import os

def count_folders_in_directory(directory_path):
    """Counts the number of folders in the specified directory."""
    try:
        # List all items in the directory and filter only folders
        folders = [f for f in os.listdir(directory_path) if os.path.isdir(os.path.join(directory_path, f))]
        folder_count = len(folders)
        print(f"Total number of folders in '{directory_path}': {folder_count}")
        return folder_count
    except FileNotFoundError:
        print(f"Error: Directory '{directory_path}' not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

# Specify the directory path
directory_path = r"GenAI - Uber Creatives"

# Count folders
count_folders_in_directory(directory_path)


Total number of folders in 'GenAI - Uber Creatives': 3


3

In [ ]:
import os

def count_files_by_type_in_directory(directory_path, file_extensions):
    """Counts the number of files with specified extensions across all folders and subfolders."""
    try:
        # Dictionary to store the overall count for each file extension
        overall_counts = {ext: 0 for ext in file_extensions}

        # Walk through the directory and all its subdirectories
        for root, dirs, files in os.walk(directory_path):
            # Count files by extension in the current folder (root)
            for file in files:
                for ext in file_extensions:
                    if file.lower().endswith(ext):
                        overall_counts[ext] += 1

        # Display the overall counts
        print(f"Overall file counts in '{directory_path}':")
        for ext, count in overall_counts.items():
            print(f"  {ext.upper()} files: {count}")

        return overall_counts
    except FileNotFoundError:
        print(f"Error: Directory '{directory_path}' not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

# Specify the directory path and file extensions to count
directory_path = r"GenAI - Uber Creatives"
file_extensions = ['.jpg', '.png', '.jpeg', '.gif', '.bmp', '.mp4', '.avi']

# Count files by type in the directory and all its subfolders
count_files_by_type_in_directory(directory_path, file_extensions)


Overall file counts in 'GenAI - Uber Creatives':


{'.jpg': 52,
 '.png': 1120,
 '.jpeg': 0,
 '.gif': 14,
 '.bmp': 0,
 '.mp4': 206,
 '.avi': 0}

In [32]:
import re
import pandas as pd
import json
import os
import uuid
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance
import cv2
import numpy as np
import requests
import base64
from requests_toolbelt import MultipartEncoder

To run Qdrant: docker run -p 6333:6333 -p 6334:6334 -v %cd%/qdrant_storage:/qdrant/storage qdrant/qdrant

In [ ]:
client = OpenAI(api_key="")
# qdrant_client = QdrantClient("localhost", port=6333)
qdrant_client = QdrantClient(url = 'http://3.101.65.253:6333')

In [35]:
# Normalization function
def normalize_campaign_name(name):
    """Normalize campaign names for comparison."""
    if not name:
        return ""
    
    # Replace slashes with underscores and hyphens with underscores for consistency
    name = name.replace('/', '_').replace('-', '_')
    
    # Replace apostrophes with underscores
    name = name.replace("'", "_")
    
    # Normalize dates (e.g., 2-29-24 -> 2/29/24)
    name = re.sub(r'(\d{1,2})-(\d{1,2})-(\d{2,4})', r'\1/\2/\3', name)
    
    # Replace multiple underscores with a single underscore
    name = re.sub(r'_+', '_', name)
    
    # Remove trailing underscores
    name = name.strip('_')
    
    # Convert to lowercase
    return name.lower()

# Function to find matching campaigns in Excel

def find_matching_campaign(normalized_folder, df):
    for _, row in df.iterrows():
        normalized_excel = row['Normalized Campaign Name']
        if normalized_folder == normalized_excel or normalized_folder in normalized_excel or normalized_excel in normalized_folder:
            return row.to_dict()
    return None


# def normalize_campaign_name(name):
#     # Convert slashes to hyphens
#     name = name.replace('/', '-')
    
#     # Remove date patterns (both slash and hyphen formats)
#     name = re.sub(r'\d{1,2}[-]\d{1,2}[-]\d{2,4}', '', name)
    
#     # Remove year patterns at the end (e.g., /24)
#     name = re.sub(r'[-/]\d{2}$', '', name)
    
#     # Replace underscores, spaces, and multiple hyphens with a single underscore
#     name = re.sub(r'[_\s-]+', '_', name)
    
#     # Remove any trailing underscores
#     name = name.strip('_')
    
#     # Convert to lowercase
#     return name.lower()

# def find_matching_campaign(normalized_folder, df):
#     for _, row in df.iterrows():
#         normalized_excel = row['Normalized Campaign Name']
#         if normalized_folder == normalized_excel or normalized_folder in normalized_excel or normalized_excel in normalized_folder:
#             return row.to_dict()
#     return None

In [36]:
# Remove backticks and the word 'json'
def clean_json_response(response):
    # Remove backticks and the word 'json'
    cleaned = re.sub(r'```json?|```', '', response)
    # Strip any leading or trailing whitespace
    cleaned = cleaned.strip()
    return cleaned

In [62]:
# The OG
def call_chatgpt(frames, is_video, campaign_folder, brand, target):
    # Prepare the prompt to ask GPT-4 about seasons and holidays
    prompt = f"""Return output in JSON format. If you are unsure about anything, leave it as an empty string.
        Industry: [Specify the industry]
        Company: [Provide the company name]
        Brand: {brand}
        Brand Type: [Specify brand type (e.g., luxury, service)]
        Brand Mascots: [List any brand mascots]
        Product/Service: [Describe the advertised product or service]
        Product Category: [Specify product category]
        Business Category: [Specify business category]
        Ad Objective: [State the primary goal (like Brand Awareness, Lead Generation, Sales Promotion, Customer Retention)]
        Format: [Specify the ad format(Image, Video, Interactive, Game)]
        Targeting : [What kind of ad targeting methods is used(can be none): {target}]
        Target Market: [Specify the target market for the product/service]
        Target Audience: [Specify demographic or psychographic details]
        Key Message: [Summarize the main takeaway in one sentence]
        Tone and Mood: [Describe the overall atmosphere]
        Start Date: [Extract Start Date from Folder Name: {campaign_folder}. Example 1: SVT_ChurchillDowns_US_Twinspires_01-26-24-01-27-24_JourneyAd extract 01-26-2024. Example 2:Evidens De Beaute_10/04 extract 10-04-2024]
        End Date: [Extract End Date from Folder Name: {campaign_folder} Example 1: SVT_ChurchillDowns_US_Twinspires_01-26-24-01-27-24_JourneyAd extract 01-27-24. Example 2:Evidens De Beaute_10/04 extract NULL]]
        Weekends: [Count number of weekends between the start and end date]
        Holidays: [Specify holidays that occurr around the start and end dates(Leave as Empty string if none found)]
        National Events: [Specify US national events that occurr around the start and end dates(Leave as Empty string if none found)]
        International Events: [Specify relevant international events that occurr around the start and dates(Leave as Empty string if none found)]
        Sport Events: [Specify relevant sports events like Super Bowl that occurr around the start and end dates(Leave as Empty string if none found)]
        Creative Theme: [Describe the creative theme(Humor, Aspirational, Relatable, Cause-Related Marketing, Futuristic, Problem-Solving, Throwback Themes, Minimalism, Trendy, Bold Imagery)]
        Strategy: [Describe the advertising strategy - Emotional Appeal, Lifestyle, Social Responsibility, Innovation and Technology, Nostalgia, Simplicity, Cultural Relevance, Visual Impact]
        Visual Elements:
        Setting: [Describe the primary location(s)]
        Characters: [List main people or animated figures]
        Colors: [Mention the two most dominant colors present in the creative as a string]
        Imagery: [Describe what is happening in the creative and note significant objects or symbols as a string]"""
    if is_video:
        prompt += f"""
                Narrative Structure:
                Opening: [Describe how the ad begins]
                Middle: [Summarize the main content]
                Closing: [Explain how the ad concludes]
                Cinematography: [Mention notable camera techniques or visual effects]"""

    prompt += f"""
            Call-to-Action: [State the specific action viewers are urged to take]
            Unique Selling Proposition: [Identify the key differentiator highlighted]
            Follow a Strict Json Format and limit your output to a maximum of 500 tokens."""


    PROMPT_MESSAGES = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                *[{"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{frame}"}} for frame in frames]
            ],
        }
    ]

    params = {
        "model": "gpt-4o-mini",
        "messages": PROMPT_MESSAGES,
        "max_tokens": 500,
    }

    result = client.chat.completions.create(**params)
    return result.choices[0].message.content 

    # Removed Audio Elements as for now we don't consider audio.
    # Music: [Describe style and mood of background music]                
    # # Voiceover: [Note presence and style of narration

In [63]:
def call_cpm_v_model(frames, is_video, campaign_folder, brand, target):
    # Your CPM-V model URL
    print("No of frames received:",frames,"isvideo:",is_video,campaign_folder, brand, target)
    CPM_V_URL = "http://a1168d1c1c09247789fdf3e8c5d8cc9d-1339720531.us-east-1.elb.amazonaws.com:8000/1/docs#/default/image_infer_image_infer_post"

    # Prepare the prompt
    prompt = f"""Return output in JSON format. If you are unsure about anything, leave it as an empty string.
        Industry: [Specify the industry]
        Company: [Provide the company name]
        Brand: {brand}
        Brand Type: [Specify brand type (e.g., luxury, service)]
        Brand Mascots: [List any brand mascots]
        Product/Service: [Describe the advertised product or service]
        Product Category: [Specify product category]
        Business Category: [Specify business category]
        Ad Objective: [State the primary goal (like Brand Awareness, Lead Generation, Sales Promotion, Customer Retention)]
        Format: [Specify the ad format(Image, Video, Interactive, Game)]
        Targeting : [What kind of ad targeting methods is used(can be none): {target}]
        Target Market: [Specify the target market for the product/service]
        Target Audience: [Specify demographic or psychographic details]
        Key Message: [Summarize the main takeaway in one sentence]
        Tone and Mood: [Describe the overall atmosphere]
        Start Date: [Extract Start Date from Folder Name: {campaign_folder}. Example 1: SVT_ChurchillDowns_US_Twinspires_01-26-24-01-27-24_JourneyAd extract 01-26-2024. Example 2:Evidens De Beaute_10/04 extract 10-04-2024]
        End Date: [Extract End Date from Folder Name: {campaign_folder} Example 1: SVT_ChurchillDowns_US_Twinspires_01-26-24-01-27-24_JourneyAd extract 01-27-24. Example 2:Evidens De Beaute_10/04 extract NULL]]
        Weekends: [Count number of weekends between the start and end date]
        Holidays: [Specify holidays that occurr around the start and end dates(Leave as Empty string if none found)]
        National Events: [Specify US national events that occurr around the start and end dates(Leave as Empty string if none found)]
        International Events: [Specify relevant international events that occurr around the start and dates(Leave as Empty string if none found)]
        Sport Events: [Specify relevant sports events like Super Bowl that occurr around the start and end dates(Leave as Empty string if none found)]
        Creative Theme: [Describe the creative theme(Humor, Aspirational, Relatable, Cause-Related Marketing, Futuristic, Problem-Solving, Throwback Themes, Minimalism, Trendy, Bold Imagery)]
        Strategy: [Describe the advertising strategy - Emotional Appeal, Lifestyle, Social Responsibility, Innovation and Technology, Nostalgia, Simplicity, Cultural Relevance, Visual Impact]
        Visual Elements:
        Setting: [Describe the primary location(s)]
        Characters: [List main people or animated figures]
        Colors: [Mention the two most dominant colors present in the creative as a string]
        Imagery: [Note significant objects or symbols as a string]"""
    if is_video:
        prompt += f"""
                Narrative Structure:
                Opening: [Describe how the ad begins]
                Middle: [Summarize the main content]
                Closing: [Explain how the ad concludes]
                Cinematography: [Mention notable camera techniques or visual effects]"""

    prompt += f"""
            Call-to-Action: [State the specific action viewers are urged to take]
            Unique Selling Proposition: [Identify the key differentiator highlighted]
            Follow a Strict Json Format and limit your output to a maximum of 500 tokens."""
    form_data = {}
    for i, frame in enumerate(frames):
        form_data[f'image_files'] = (f'frame_{i}.jpg', base64.b64decode(frame), 'image/jpeg')

    frame_indices = list(range(len(frames)))
    messages = [{"type": "text", "role": "user", "content": [0] + frame_indices + [prompt]}]

    form_data['json_messages'] = (None, json.dumps(messages), 'application/json')
    form_data['json_kwargs'] = (None, json.dumps({"temperature": 0.5}), 'application/json')

    # Create multipart encoder
    multipart_data = MultipartEncoder(fields=form_data)

    # Prepare headers
    headers = {
        'Content-Type': multipart_data.content_type,
        'accept': 'application/json'
    }

    # Make the POST request
    response = requests.post(CPM_V_URL, headers=headers, data=multipart_data)

    if response.status_code == 200:
        result = response.json()
        return result
    else:
        raise Exception(f"Error calling CPM-V model: {response.status_code} - {response.text}")

In [64]:
def resize_frame(frame, scale_percent=50):
    """Resize the frame by a percentage."""
    width = int(frame.shape[1] * scale_percent / 100)
    height = int(frame.shape[0] * scale_percent / 100)
    return cv2.resize(frame, (width, height))

def process_video(video_path, scale_percent=50):
    video = cv2.VideoCapture(video_path)
    base64Frames = []
    prev_hist = None
    frame_threshold = 0.8

    def calculate_histogram(frame):
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        hist = cv2.calcHist([hsv], [0, 1], None, [180, 256], [0, 180, 0, 256])
        cv2.normalize(hist, hist, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)
        return hist.astype(np.float32).flatten()  # Ensure histogram is float32

    def is_keyframe(frame, prev_hist, threshold):
        if prev_hist is None:
            return True
        curr_hist = calculate_histogram(frame)

        # Ensure both histograms have the same shape
        if curr_hist.shape != prev_hist.shape:
            return True  # Treat as keyframe if shapes don't match

        diff = cv2.compareHist(prev_hist.reshape(-1, 1), curr_hist.reshape(-1, 1), cv2.HISTCMP_CORREL)
        return diff < threshold

    frame_count = 0
    while video.isOpened():
        success, frame = video.read()
        if not success:
            break
        
        # Resize the frame before processing
        frame = resize_frame(frame, scale_percent)

        frame_count += 1
        if frame_count % 5 == 0:  # Process every 5th frame
            if is_keyframe(frame, prev_hist, frame_threshold):
                _, buffer = cv2.imencode(".jpg", frame)
                base64Frames.append(base64.b64encode(buffer).decode("utf-8"))
                prev_hist = calculate_histogram(frame)

    video.release()
    return base64Frames


def process_image(image_path, scale_percent=50):
    """Resize the image before encoding it to base64."""
    image = cv2.imread(image_path)
    if image is not None:
        # Resize the image
        image = resize_frame(image, scale_percent)
        
        # Encode to base64
        _, buffer = cv2.imencode(".jpg", image)
        return base64.b64encode(buffer).decode("utf-8")
    else:
        raise FileNotFoundError(f"Image file not found: {image_path}")
    

In [65]:

def read_excel_to_dataframe(file_path, sheet_name):
    df = pd.read_excel(file_path, sheet_name=sheet_name)
    return df

In [66]:
def embed_text(text):
    response = client.embeddings.create(
        model="text-embedding-ada-002",
        input=text
    )
    print("Embedded Text")
    return response.data[0].embedding

def store_in_qdrant(collection_name, vector, payload):

    qdrant_client.upsert(
        collection_name=collection_name,
        points=[
            {
                'id': payload['id'],
                'vector': vector,
                'payload': payload
            }
        ]
    )
    return 

In [67]:
def get_file_info(file_path):
    file_size = os.path.getsize(file_path)
    _, file_extension = os.path.splitext(file_path)
    # Normalize the file extension to lowercase
    file_extension = file_extension.lower()
    # Determine the nature based on the file extension
    if file_extension == '.gif':
        nature = "video"
    elif file_extension in ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']:
        nature = "image"
    elif file_extension in ['.mp4', '.avi', '.mkv', '.mov']:
        nature = "video"
    else:
        nature = "unknown"
    return {
        "size": file_size,
        "nature": nature
    }

In [68]:
import hashlib
def calculate_md5(file_path):
    hash_md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

In [ ]:
def clean_payload_data(data):
    def lowercase_dict(d):
        return {
            k.lower(): (
                lowercase_dict(v) if isinstance(v, dict)
                else v.lower() if isinstance(v, str)
                else v
            )
            for k, v in d.items()
        }
    return lowercase_dict(data)

def process_file(file_path, campaign_folder, excel_data, model):
    file_info = get_file_info(file_path)
    is_video = file_info['nature'] == 'video'
    
    print("Processing...",file_info['nature'], file_path)
    frames = process_video(file_path) if is_video else [process_image(file_path)]

    brand,target = excel_data['Brand'],excel_data['Targeting']
    
    
    if model == "cpm-v":
        print("Making cpm call")
        model_response = call_cpm_v_model(frames, is_video, campaign_folder, brand, target)
    elif model == "gpt-4o-mini":
        model_response = call_chatgpt(frames, is_video, campaign_folder, brand, target)
    else: 
        model_response = call_chatgpt(frames, is_video, campaign_folder, brand, target)
    
    print("Model response:",model_response)
    cleaned_data = clean_json_response(model_response)
    print(cleaned_data)
    try:
        chatgpt_data = json.loads(cleaned_data)
    except json.JSONDecodeError as e:
        print(f"JSON Decode Error: {e}")
        print(f"Error occurred at position: {e.pos}")
        print(f"Problematic JSON:\n{cleaned_data}")
        print(f"Substring around error:\n{cleaned_data[max(0, e.pos-50):e.pos+50]}")
    
    combined_data = {**chatgpt_data, **excel_data}
    
    combined_data['file_size'] = file_info['size']
    combined_data['file_type'] = file_info['nature']
    combined_data['md5_hash'] = calculate_md5(file_path)
    combined_data['attention'] = ''
    combined_data['engagement'] = ''
    duration = combined_data.get('Duration(days)', 0)
    if (combined_data.get('booked measure',0) < combined_data.get('delivered measure',0)) and  combined_data.get('conversion', 0)>0.03:
        combined_data['engagement'] = 'high'
        combined_data['attention'] = 'high'
    elif  (combined_data.get('booked measure',0) < combined_data.get('delivered measure',0)) and  combined_data.get('conversion', 0)<0.03:
        combined_data['engagement'] = 'moderate'
        combined_data['attention'] = 'high'
    elif  (combined_data.get('booked measure',0) > combined_data.get('delivered measure',0)) and  combined_data.get('conversion', 0)>0.03:
        combined_data['engagement'] = 'moderate'
        combined_data['attention'] = 'moderate'
    elif  (combined_data.get('booked measure',0) > combined_data.get('delivered measure',0)) and  combined_data.get('conversion', 0)<0.03:
        combined_data['engagement'] = 'moderate'
        combined_data['attention'] = 'high'

    combined_data['duration_category'] = 'short' if duration <= 7 else 'medium' if duration <= 30 else 'long'
    
    combined_data = clean_payload_data(combined_data)
    combined_data.pop('unnamed: 20')
    # vector_text = f"""{combined_data.get('industry', '')} {combined_data.get('product/service', '')} {combined_data.get('ad objective','')} {combined_data.get('key message', '')} {combined_data.get('tone and mood','')}  {combined_data.get('brand', '')}  {combined_data.get('targeting', '')}  {combined_data.get('seasonal/holiday elements', '')}  {combined_data.get('duration_category', '')} {combined_data.get('brand type', '')}"""
    

    vector_text = f"""{combined_data.get('industry', '')} {combined_data.get('product/service', '')} {combined_data.get('ad objective','')} {combined_data.get('key message', '')} {combined_data.get('tone and mood','')} {combined_data.get('brand', '')} {combined_data.get('targeting', '')} {combined_data.get('seasonal/holiday elements', '')} {combined_data.get('duration_category', '')} {combined_data.get('brand type', '')} {combined_data.get('creative theme', '')} {combined_data.get('strategy', '')} {combined_data.get('target market', '')} {combined_data.get('target audience', '')} {combined_data.get('call-to-action', '')} {combined_data.get('unique selling proposition', '')} {combined_data.get("visual elements", {}).get("colors", "")}{combined_data.get("visual elements", {}).get("imagery", "")} {combined_data.get('end_date', '')} {combined_data.get('business_category', '')} {combined_data.get('format', '')}"""



    print("vector text:",vector_text)
    vector = embed_text(vector_text)
    
    return vector, combined_data

In [70]:
def create_payload(campaign_folder, filename, combined_data):
    return {
        'id': str(uuid.uuid4()), # Generate a random UUID
        'file_name': filename,
        'sl no':combined_data['sl no'],
        'start_date':combined_data.get('start date',''),
        'end_date':combined_data.get('end date',''),
        'campaign_folder': campaign_folder,
        'company':combined_data.get('company',''),
        'industry': combined_data.get('industry', ''),
        'brand': combined_data.get('brand', ''),
        'brand_type': combined_data.get('brand type',''),
        'product_service': combined_data.get('product/service', ''),
        'business_category':combined_data.get('business category', ''),
        'ad_objective': combined_data.get('ad objective', ''),
        'file_size': combined_data['file_size'],
        'file_type': combined_data['file_type'],
        'type': combined_data.get('type',''),
        'booked_measure_impressions': combined_data.get('booked measure',0),
        'delivered_measure_impressions':combined_data.get('delivered measure',0),
        'duration(days)': combined_data.get('duration(days)',0),
        'duration_category': combined_data.get('duration_category',''),
        'holidays': combined_data.get('holidays',''),
        'national_events': combined_data.get('national events',''),
        'international_events':combined_data.get('international events',''),
        'weekends':combined_data.get('weekends',0),
        'sport_events':combined_data.get('sport events',''),
        'creative_theme':combined_data.get('creative theme',''),
        'strategy':combined_data.get('strategy',''),
        'colors':combined_data.get("visual elements", {}).get("colors", ""),
        'imagery':combined_data.get("visual elements", {}).get("imagery", ""),
        'targeting': combined_data.get('targeting', ''),
        'md5_hash':combined_data.get('md5_hash',''),
        'key_message': combined_data.get('key message', ''),
        'tone_mood': combined_data.get('tone and mood', ''),
        'clicks': combined_data.get('clicks', 0),
        'conversion': combined_data.get('conversion', 0),
        'attention': combined_data.get('attention', ''),
        'engagement': combined_data.get('engagement', ''),
        'full_data': json.dumps(combined_data)
    }

In [71]:

def store_data(store_function, *args):
    store_function(*args)


In [72]:
def process_directory(root_dir, excel_path, excel_sheet, store_function,model):
    # For every folder present in the directory
    # Find the matchinf excel line
    # Then for every creative present in the directory
        # Process
        # Format as Required
        # Store as Required.
    df = read_excel_to_dataframe(excel_path, excel_sheet)
    df.drop('Creative (GDrive)',axis=1, inplace=True)
    df.drop('Campaign Objectives',axis=1, inplace=True)
    df.drop('ROAS',axis=1, inplace=True)
    # Normalize campaign names in the    DataFrame once
    df['Normalized Campaign Name'] = df['Campaign Name'].apply(normalize_campaign_name)

    
    for campaign_folder in os.listdir(root_dir):
        campaign_path = os.path.join(root_dir, campaign_folder)
        if not os.path.isdir(campaign_path):
            continue
        
        normalized_folder = normalize_campaign_name(campaign_folder)
        excel_data = find_matching_campaign(normalized_folder, df)
        
        if excel_data is None:
            print(f"No matching campaign found for folder: {campaign_folder}")
            continue
        print(f"Found match for: {campaign_folder}")
        
        for root, dirs, files in os.walk(campaign_path):
            for filename in files:
                if filename.lower().endswith(('.jpg', '.png', '.mov', '.mp4', '.gif')):
                    file_path = os.path.join(root, filename)
                    print(f"Processing: {file_path}")
                    
                    vector, combined_data = process_file(file_path, campaign_folder, excel_data, model)
                    payload = create_payload(campaign_folder, filename, combined_data)
                    store_data(store_function, vector, payload)
                    
                    print(f"Processed and stored data for: {filename}, With ID: {payload['id']}")

    print("Data processing and storage complete.")

    
# Example usage for Qdrant
def qdrant_store_function(vector, payload):
    collection_name = "Media_Performance"
    store_in_qdrant(collection_name, vector, payload)
    return 
# Initialize Qdrant collection
def init_qdrant():
    collection_name = "Media_Performance"
    collections = qdrant_client.get_collections()
    if collection_name in collections:
        print(f"Collection '{collection_name}' already exists. Skipping recreation.")
    else:
        # Create a new collection if it doesn't exist
        qdrant_client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
        )
        print(f"Collection '{collection_name}' created successfully.")
    return

In [73]:
# Main execution
if __name__ == "__main__":
    root_directory = "C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/GenAI - Uber Creatives"
    excel_path = 'GenAI - Uber Creatives/Creatives.xlsx'
    excel_sheet = 'Data'
    model = "gpt-4o-mini"
    # model = "cpm-v"
    
    # init_qdrant()
    process_directory(root_directory, excel_path,excel_sheet, qdrant_store_function,model)

Found match for: CPG_ConstellationBrands_US_Cinco_5-1-24-5-5-24_JourneyAd
Processing: C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/GenAI - Uber Creatives\CPG_ConstellationBrands_US_Cinco_5-1-24-5-5-24_JourneyAd\FINAL CINCO CREATIVE\CoronaCincoDispatch_1.png
Processing... image C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/GenAI - Uber Creatives\CPG_ConstellationBrands_US_Cinco_5-1-24-5-5-24_JourneyAd\FINAL CINCO CREATIVE\CoronaCincoDispatch_1.png
Model response: ```json
{
    "Industry": "Beverage",
    "Company": "Constellation Brands",
    "Brand": "Constellation Brands",
    "Brand Type": "Alcoholic Beverage",
    "Brand Mascots": "",
    "Product/Service": "Corona De Mayo beer",
    "Product Category": "Alcoholic Beverage",
    "Business Category": "Consumer Goods",
    "Ad Objective": "Brand Awareness",
    "Format": "Image",
    "Targeting": "nan",
    "Target Market": "Beer drinkers, particularly during Cinco de Mayo",
    "Target Audience": 

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/GenAI - Uber Creatives\\SVT_Apple_US_ApplePayPIP_04-22-24-05-31-24_JourneyAd\\04_Creative Assets\\Phase 1\\Apple Pay_Pay Holiday Everyday Wave 2 Q124 US_Uber_1146x393_JourneyAd_US_Ultrawide_Text On White_1x_NA.png'

In [82]:
# Main execution
if __name__ == "__main__":
    root_directory = "C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/GenAI - Uber Creatives"
    excel_path = 'GenAI - Uber Creatives/Creatives.xlsx'
    excel_sheet = 'Data'
    model = "gpt-4o-mini"
    # model = "cpm-v"
    
    # init_qdrant()
    process_directory(root_directory, excel_path,excel_sheet, qdrant_store_function,model)

Found match for: SVT_VH1_US_Celebritysquares_10-30-23_11-17-23_JourneyAd
Processing: C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/GenAI - Uber Creatives\SVT_VH1_US_Celebritysquares_10-30-23_11-17-23_JourneyAd\Creatives\Enroute\Celebrity-enroute-gam_v1.png
Processing... image C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/GenAI - Uber Creatives\SVT_VH1_US_Celebritysquares_10-30-23_11-17-23_JourneyAd\Creatives\Enroute\Celebrity-enroute-gam_v1.png
Model response: ```json
{
  "Industry": "Entertainment",
  "Company": "VH1",
  "Brand": "VH1",
  "Brand Type": "Service",
  "Brand Mascots": "",
  "Product/Service": "Celebrity Squares game show",
  "Product Category": "Television Programming",
  "Business Category": "Media",
  "Ad Objective": "Brand Awareness",
  "Format": "Video",
  "Targeting": "nan",
  "Target Market": "General television viewers",
  "Target Audience": "Adults aged 18-49",
  "Key Message": "Join the fun and excitement of Celebrity Squares 

### Indexing

In [83]:
# Setting up Indexing....
from qdrant_client import QdrantClient

client = QdrantClient(url="http://3.101.65.253:6333/")
numeric_fields = [
    "sl no","file_size", "booked_measure_impressions", "delivered_measure_impressions",
   "clicks", "conversion",  "duration(days)", 
]
for field in numeric_fields:
    client.create_payload_index(
        collection_name="Media_Performance",
        field_name=field,
        field_schema="float"
    )
    
text_fields = [
    "id", "file_name", "campaign_folder", "file_type", "type", "industry","ad_objective", "tone_mood", "product_service", "targeting","duration_category", "brand","business_category", "brand type", "company", "creative_theme","strategy"
]
for field in text_fields:
    client.create_payload_index(
        collection_name="Media_Performance",
        field_name=field,
        field_schema="keyword"
    )

### Test Matching

In [ ]:
def test_mapping(root_dir, excel_path, excel_sheet):
    df = read_excel_to_dataframe(excel_path, excel_sheet)
    all_matched = True
    unmatched_folders = []

    # Normalize all campaign names in the DataFrame
    df['Normalized Campaign Name'] = df['Campaign Name'].apply(normalize_campaign_name)

    for campaign_folder in os.listdir(root_dir):
        campaign_path = os.path.join(root_dir, campaign_folder)
        if not os.path.isdir(campaign_path):
            continue

        normalized_folder = normalize_campaign_name(campaign_folder)
        excel_data = None

        # Try to find a match using normalized names
        for _, row in df.iterrows():
            if normalized_folder in row['Normalized Campaign Name'] or row['Normalized Campaign Name'] in normalized_folder:
                excel_data = row.to_dict()
                break

        if excel_data is None:
            all_matched = False
            unmatched_folders.append(campaign_folder)
            print(f"No matching campaign found for folder: {campaign_folder}")
            print(f"  Normalized folder name: {normalized_folder}")
        else:
            print(f"Match found for folder: {campaign_folder}")
            print(f"  Normalized folder name: {normalized_folder}")
            print(f"  Matched with: {excel_data['Campaign Name']}")
            print(f"  Normalized Excel name: {excel_data['Normalized Campaign Name']}")

        for root, dirs, files in os.walk(campaign_path):
            for filename in files:
                if filename.lower().endswith(('.jpg', '.png', '.mov', '.mp4', '.gif')):
                    file_path = os.path.join(root, filename)
                    print(f"  File would be processed: {file_path}")

    if all_matched:
        print("\nAll folders successfully matched with Excel data.")
    else:
        print("\nWarning: Some folders could not be matched:")
        for folder in unmatched_folders:
            print(f"  - {folder}")

    return all_matched

root_directory = "C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/GenAI - Uber Creatives"
excel_path = 'GenAI - Uber Creatives/Creatives.xlsx'
excel_sheet = 'Data'
test_mapping(root_directory,excel_path,excel_sheet)

### Testing FastAPI Server

In [ ]:
url = 'http://127.0.0.1:8000/'

response  = requests.get(url)

if response.status_code == 200:
    # Parse the JSON response
    result = response.json()
    
    print("Search Results:",result)

else:
    print(f"Error: {response.status_code}")
    print(response.text)

In [ ]:
import requests

url = 'http://127.0.0.1:8000/search'

query = {
    "text": "",
    "filters": {},  # Add any specific filters if needed
    "min_conversions": 0.001,  # Example: Consider campaigns with very low conversion rates
    "max_clicks": 1000,  # Example: Limit to campaigns with fewer clicks
    "file_type": None,  # Set to specific type if needed
    "duration_category": None,
    "industry": None,
    "brand_type": None,
    "ad_objective": None,
    "tone_mood": None,
    "max_impressions": 100000  # Example: Limit to campaigns with fewer impressions
}

response = requests.post(url, json=query)

if response.status_code == 200:
    result = response.json()
    
    search_results = result["search_results"]
    generated_response = result["generated_response"]
    
    print("Search Results:")
    for ad in search_results["results"]:
        print(f"- ID: {ad['id']}")
        print(f"  File Name: {ad['file_name']}")
        print(f"  Brand: {ad['brand']}")
        print(f"  Product/Service: {ad['product_service']}")
        print(f"  Conversions: {ad['conversion']}")
        print(f"  Clicks: {ad['clicks']}")
        print(f"  Impressions: {ad['delivered_measure_impressions']}")
        print(f"  Ad Objective: {ad['ad_objective']}")
        print(f"  Tone/Mood: {ad['tone_mood']}")
        print("  ---")
    
    print("\nGenerated Response:")
    print(generated_response)
else:
    print(f"Error: {response.status_code}")
    print(response.text)

## In house Model Experiment

In [ ]:
import requests
import json
from PIL import Image

url = "http://a1168d1c1c09247789fdf3e8c5d8cc9d-1339720531.us-east-1.elb.amazonaws.com:8000/1/image_infer"

prompt ='can yo help me decide which is better?'
json_messages = json.dumps([{"type": "text", "role": "user", "content": [0,1,0,prompt]}])

json_kwargs = json.dumps({"temperature": 0.5})
image_path1 = 'GenAI - Uber Creatives/GenAI - Uber Creatives/CPG_Jagermeister_US_Events_5-2-24-5-12-24_JourneyAd/GAM Creatives/Version 1/Enroute.png'
image_path2 = 'GenAI - Uber Creatives/GenAI - Uber Creatives/CPG_Jagermeister_US_Events_5-2-24-5-12-24_JourneyAd/GAM Creatives/Version 1/OnTrip.png'
image_files = {
    'image_files': ('Enroute.png', open(image_path1, 'rb'), 'image/jpeg'),
    'image_files': ('OnTrip.png', open(image_path2, 'rb'), 'image/jpeg'),
}

data = {
    
    'json_messages': json_messages,
    'json_kwargs': json_kwargs
}

# Make the POST request
response = requests.post(url, files=image_files, data=data)

# Check the response
if response.status_code == 200:
    print("Request successful!")
    print(response.json())
else:
    print(f"Request failed with status code: {response.status_code}")
    print(response.text)

## Media Campaign What Questions are not present

In [ ]:
root_directory = "C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/GenAI - Uber Creatives"
excel_path = 'GenAI - Uber Creatives/Creatives.xlsx'


import pandas as pd

In [ ]:
def test_mapping(root_dir, excel_path, excel_sheet):
    df = read_excel_to_dataframe(excel_path, excel_sheet)
    all_matched = True
    unmatched_folders = []

    # Normalize all campaign names in the DataFrame
    df['Normalized Campaign Name'] = df['Campaign Name'].apply(normalize_campaign_name)

    for campaign_folder in os.listdir(root_dir):
        campaign_path = os.path.join(root_dir, campaign_folder)
        if not os.path.isdir(campaign_path):
            continue

        normalized_folder = normalize_campaign_name(campaign_folder)
        excel_data = None

        # Try to find a match using normalized names
        for _, row in df.iterrows():
            if normalized_folder in row['Normalized Campaign Name'] or row['Normalized Campaign Name'] in normalized_folder:
                excel_data = row.to_dict()
                break

        if excel_data is None:
            all_matched = False
            unmatched_folders.append(campaign_folder)
            print(f"No matching campaign found for folder: {campaign_folder}")
            print(f"  Normalized folder name: {normalized_folder}")
        else:
            print(f"Match found for folder: {campaign_folder}")
            print(f"  Normalized folder name: {normalized_folder}")
            print(f"  Matched with: {excel_data['Campaign Name']}")
            print(f"  Normalized Excel name: {excel_data['Normalized Campaign Name']}")

            # Get list of files from Excel
            excel_files = [file.strip() for file in excel_data.get('File Names', '').split(',') if file.strip()]
            folder_files = set()

            for root, dirs, files in os.walk(campaign_path):
                for filename in files:
                    if filename.lower().endswith(('.jpg', '.png', '.mov', '.mp4', '.gif')):
                        file_path = os.path.join(root, filename)
                        print(f"  File found in folder: {file_path}")
                        folder_files.add(filename)

            # Check for files in Excel but not in folder
            missing_files = set(excel_files) - folder_files
            if missing_files:
                print("  Files in Excel but not found in folder:")
                for file in missing_files:
                    print(f"    - {file}")

    if all_matched:
        print("\nAll folders successfully matched with Excel data.")
    else:
        print("\nWarning: Some folders could not be matched:")
        for folder in unmatched_folders:
            print(f"  - {folder}")

    return all_matched

# Rest of the code remains the same
root_directory = "C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/GenAI - Uber Creatives"
excel_path = 'GenAI - Uber Creatives/Creatives.xlsx'
excel_sheet = 'Data'
test_mapping(root_directory, excel_path, excel_sheet)

### Test a prompt

In [ ]:
prompt = "You are a relevance filter part of a multi agent chat system. Given the entire conversation history of the session along with the current user query, you are to return an enhanced query and selected portions of the conversation history which you think are relevant for this query. The agent that receives your output can do the following - 1. Make a media plan 2. Provide campaign performance reports 3. Provide Creative insights 4. Provide constructive feedback based on creatives. 5. Show how similar creatives performed 6. Trend Analysis based on historical data. So enhance the query appropriately. You try to minimize tokens used by not using spaces between objects"
query = """
 [{'role': 'system', 'content': 'You are an AI agent that determines required_outcomes, check for relatedness of query and extract parameters for filtering.\nRequired Outcomes:\nRepresented as an array of integers based on the following mapping:\n1: Media Plan\n2: Analysis of Trends\n3: Campaign Performance\n4: Creative Insights\n5: Performance Summary - Summary of other outputs\nunrelated_query: Check if the current Query is related to the conversation history.\nparameters: Extract season, holiday, industry, duration_category and brand if present in query.\nExamples -\n1.Conversation_history: "" \n  User: "Generate a media plan for a shoe brand"\nResponse:{\n  "required_outcomes": [1, 2, 3, 4, 5],\n  "unrelated_query": true\n  "parameters":{\n    "industry":"fashion"\n  }\n}\n2.Conversation_history: "" \nUser: "Show me summer campaigns for tech brands"\nResponse:\n{\n  "required_outcomes": [2, 3, 5],\n  "unrelated_query": true,\n  "parameters": {\n    "season": "summer",\n    "industry": "tech",\n  }\n}\n3. Conversation_history: "" \nUser: "Can you provide campaign performance for a fashion brand"\nResponse:\n{\n  "required_outcomes": [3, 5],\n  "unrelated_query": true,\n  "parameters": {\n    "industry": "fashion retail"\n  }\n}\n\n4. User: "What are some creative ideas for a spring campaign in the beauty industry?"\nResponse:\n{\n  "required_outcomes": [4],\n  "unrelated_query": true,\n  "parameters": {\n    "season": "spring",\n    "industry": "beauty"\n  }\n}\n5.Conversation_history: ""  \nUser: "Show me creative inspiration for christmas campaigns in the food industry and how similar campaigns performed"\nResponse:\n{\n  "required_outcomes": [4, 3, 2, 5],\n  "unrelated_query": true,\n  "parameters": {\n    "holiday": "christmas",\n    "industry": "food"\n  }\n}\n6. Conversation_history: "" \nUser: "Help me hack into NASA"\nResponse:\n{\n  "required_outcomes": [],\n  "unrelated_query": true,\n}\n7.  Conversation_history: "User: Generate media plan to sell fashion brands..."\n  User: "Generate a media plan for a dog food brand"\n  Response:{\n  "required_outcomes": [1, 2, 3, 4, 5],\n  "unrelated_query": true\n  "parameters":{\n    "industry":"pet care"\n  }\n}'}, {'role': 'user', 'content': 'Can you provide campaign performance feedback for the same'}, {'role': 'system', 'content': 'Found and filtered some relevant data. Searching for additional data most relevant to your query...\nIncorporating additional data to your response.\n## Campaign Performance Report\n\n**Brand and Product:** H&M - The Festive Collection  \n**Campaign Objective:** Sales Promotion  \nThe budget was calculated based on the total spend of $2369.44 for the campaign duration, which was allocated to maximize impressions and engagement during the festive season.\n\n| Campaign Duration                 |       Budget         | Booked Impressions    | Delivered Impressions | Clicks/Actions  | Value to Money (Conversions) |                ECPM                  |\n|-----------------------------------|----------------------|-----------------------|-----------------------|-----------------|------------------------------|--------------------------------------| \n| Medium (30 days)                  |      $2369.44        |     125,001           |        157,963        |    5425         |        1.95%                 | $14.97                               | \n\n| Creative Snapshot                      | Creative Thumbnail                           |\n|----------------------------------------|----------------------------------------------|\n| A vibrant campaign showcasing fashionable clothing against a bright orange backdrop, featuring a female model. | ![HM-Dispatch-F1_v1.png](http://127.0.0.1:8000/static/images/33/69/33694ea64845f66d479d69507724451d.thumbnail.jpg "H&M - The Festive Collection") |\n\n---\n\n**Brand and Product:** H&M - The Festive Collection Clothing Line  \n**Campaign Objective:** Sales Promotion  \nThe budget was similarly calculated at $2369.44, aimed at driving sales through engaging visuals and a festive theme.\n\n| Campaign Duration                 |       Budget         | Booked Impressions    | Delivered Impressions | Clicks/Actions  | Value to Money (Conversions) |                ECPM                  |\n|-----------------------------------|----------------------|-----------------------|-----------------------|-----------------|------------------------------|--------------------------------------| \n| Medium (30 days)                  |      $2369.44        |     125,001           |        157,963        |    5425         |        1.95%                 | $14.97                               | \n\n| Creative Snapshot                      | Creative Thumbnail                           |\n|----------------------------------------|----------------------------------------------|\n| An elegant campaign featuring festive attire, set against a classic studio backdrop, focusing on emotional appeal. | ![HM-ontrip-F1_v1.png](http://127.0.0.1:8000/static/images/57/e3/57e3df8b4c107f8c6877cdfc28a6966b.thumbnail.jpg "H&M - The Festive Collection Clothing Line") |\n\n### Summary\nBoth campaigns effectively utilized a budget of $2369.44 over a medium duration of 30 days, achieving a significant number of impressions and clicks. The conversion rate of approximately 1.95% indicates a positive response to the festive-themed promotions, suggesting that the creative elements resonated well with the target audience. The ECPM of $14.97 reflects a cost-effective approach to reaching potential customers during the holiday season.\n## Performance Summary\n\n### Ad Objective\nThe campaigns successfully achieved the brand awareness objective by generating a high volume of impressions (157,963) and clicks (5,425). The festive theme effectively captured the audience\'s attention, leading to increased visibility for H&M\'s Festive Collection.\n\n### Call to Action\nThe click-through rate (CTR) of 3.44% (5,425 clicks from 157,963 impressions) indicates a strong engagement level. However, to improve this further, consider optimizing the call-to-action buttons to be more compelling and time-sensitive, such as \'Shop Now for Holiday Deals!\' or \'Limited Time Offer!\'\n\n### Tone and Mood\nThe festive tone of the campaigns positively impacted audience perception, creating a sense of excitement and urgency around the holiday season. The vibrant and elegant visuals aligned well with the mood, enhancing emotional engagement and encouraging potential customers to explore the collection.\n\n### Duration Category\nThe medium duration of 30 days was effective in maintaining visibility throughout the festive season. However, extending the campaign slightly before the peak shopping days could capture additional traffic and conversions, especially as consumers begin their holiday shopping earlier each year.\n\n### Unique Selling Proposition\nThe campaigns effectively highlighted seasonal offerings, showcasing H&M\'s unique festive collection. Emphasizing limited-time styles and exclusive holiday pieces could further enhance the unique selling proposition and drive urgency among consumers.\n\n### Event Context\nThe holiday season significantly influenced engagement levels, with consumers actively seeking festive attire. The campaigns capitalized on this context, leading to a higher likelihood of interaction and interest in the products being promoted.\n\n### Creative Performance\nThe creative elements were visually appealing and aligned with the festive theme. However, introducing more diversity in the models and settings could broaden the appeal and resonate with a wider audience. Additionally, incorporating user-generated content or testimonials could enhance authenticity and relatability.\n\n### Overall Performance\nOverall, the campaigns were successful in generating substantial impressions (157,963) and engagement (5,425 clicks), with a conversion rate of 1.95%. While the campaigns performed well in terms of visibility and engagement, there is room for improvement in conversion rates. Strategies such as refining the call-to-action, enhancing the urgency of offers, and diversifying creative elements could lead to better conversion outcomes.\n\n\n'}, {'role': 'user', 'content': 'Now generate a media plan for my new fashion brand.'}, {'role': 'system', 'content': 'Found and filtered some relevant data. Searching for additional data most relevant to your query...\nIncorporating additional data to your response.\n\n# Media Plan\n\n## Executive Summary\n\n| Component              | Details                                               |\n|-----------------------|-------------------------------------------------------|\n| Objectives            | Increase brand awareness for the new fashion brand.   |\n| Target Audience       | Adults aged 25-40, seeking modern gifts.              |\n| Budget                | $8,500                                               |\n| Campaign Duration     | 40 days                                             |\n\n## Target Audience\n\n| Component         | Details                                      |\n|-------------------|----------------------------------------------|\n| Age Range         | 25-40                                        |\n| Gender
| All                                          |\n| Income Level      | Information not available                    |\n| Interests         | Fashion, gifts, modern accessories           |\n| Location          | Information not available                    |\n| Behavioral Data   | Information not available                    |\n\n## Media Mix Strategy\n\n| Channel Products        | Tactics                                               | Budget Allocation | Expected Reach      | Justification                                                                 |\n|-------------------------|------------------------------------------------------|-------------------|---------------------|-------------------------------------------------------------------------------|\n| Digital Advertising      | Social media ads, display ads, and video ads.       | $5,000            | 170,000 impressions  | Digital platforms allow targeting specific demographics and interests.       |\n| Influencer Marketing     | Collaborate with fashion influencers to promote the brand. | $2,500            | 100,000 impressions  | Influencers can effectively reach the target audience and enhance brand credibility. |\n| Email Marketing          | Send newsletters and promotional offers to subscribers. | $1,000            | 30,000 impressions   | Email marketing is cost-effective and can directly engage interested consumers. |\n\n## Creative Strategy\n\n| Asset Type      | Description
                                  | Purpose                                                  | Distribution Channels                  |\n|------------------|----------------------------------------------------------|----------------------------------------------------------|----------------------------------------|\n| Visual Ads       | Sophisticated and elegant visuals featuring modern clothing and accessories. | To create a strong visual identity for the brand and attract the target audience. | Social Media, Website, Email          |\n| Video Content    | Short videos showcasing the fashion line in various settings. | To engage viewers and provide a dynamic representation of the brand. | YouTube, Instagram, Facebook          |\n\n## Measurement and Evaluation\n\n| Metric            | Description                                           | Target                  | Reporting Frequency |\n|-------------------|------------------------------------------------------|-------------------------|---------------------|\n| Impressions        | Total number of times ads are displayed to users.   | 300,000 impressions     | Weekly              |\n| Engagement Rate    | Percentage of users who interact with the ads.      | 5% engagement rate      | Weekly              |\n| Conversion Rate    | Percentage of users who make a purchase after seeing the ads. | 2% conversion rate      | Monthly             |\n\n## Overall Trends\n\n### Analysis\nCurrent trends in the fashion industry emphasize the importance of brand awareness and emotional connection with consumers. Brands are increasingly using sophisticated visuals and storytelling to engage their audience. The rise of digital platforms has also shifted focus towards video content and interactive formats that resonate with younger demographics.\n\n| Top Brands  |\n|-------------|\n| Aramis      |\n| Qi Tech     |\n\n### Campaign Strategy\n**Objective:** Increase brand awareness and drive traffic to the online store through emotional storytelling and sophisticated visuals.\n\n| Recommendation
                                                        | Supporting Insight
             |\n|--------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------|\n| Utilize a mix of emotional appeal and innovative technology in your campaigns to connect with your audience. | Brands like Aramis and Qi Tech have successfully leveraged emotional storytelling to enhance brand awareness. |\n\n### Tone and Mood\n**Description:** The campaign should convey a sophisticated and elegant tone, appealing to a fashion-forward audience.\n\n| Recommendation                                                                 | Supporting Insight
                    |\n|--------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------|\n| Maintain a consistent tone across all platforms that reflects the brand\'s identity as sophisticated and elegant. | Aramis effectively uses a sophisticated tone to resonate with its target audience.                     |\n\n### Call to Action\n**Description:** Encourage potential customers to explore the collection and engage with the brand.\n\n| Recommendation
                                                  | Supporting Insight
       |\n|--------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------|\n| Use CTAs like \'Explore the Collection Today!\' to drive immediate engagement.   | Effective CTAs from successful campaigns have shown to significantly increase conversion rates.        |\n\n### Season and Holiday\n**Description:** Align the campaign with seasonal fashion trends and events such as Fashion Week or holiday shopping seasons.\n\n| Recommendation
                                  | Supporting Insight                                                                                     |\n|--------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------|\n| Launch the campaign during peak shopping seasons to maximize visibility and engagement. | Seasonal campaigns can leverage heightened consumer interest and spending.                             |\n\n### Campaign Duration\n**Description:** A medium campaign duration to build momentum and sustain engagement.\n\n| Recommendation                                                                 | Supporting Insight                                                                                     |\n|--------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------|\n| Plan for a 4-6 week campaign duration to allow for sufficient reach and engagement. | Medium-duration campaigns have shown to maintain audience interest effectively.                        |\n\n### Booked Impressions\n**Description:** Set a target for booked impressions based on industry benchmarks and historical data.\n\n| Recommendation                                                                 | Supporting Insight
                                                     |\n|--------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------|\n| Aim for at least 500,000 booked impressions to ensure adequate brand visibility. | Successful campaigns typically achieve high impression counts to enhance brand awareness.              |\n\n### Targeting Options\n**Description:** Focus on demographic and interest-based targeting strategies to reach the ideal audience.\n\n| Recommendation                                                                 | Supporting Insight
                 |\n|--------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------|\n| Target fashion-conscious individuals aged 18-35, with interests in luxury fashion and lifestyle. | Demographic targeting has proven effective in reaching the right audience for fashion brands.         |\n\n### Conversion\n| Recommendation                                                                 | Supporting Insight
                           |\n|--------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------|\n| Monitor conversion rates closely and adjust strategies based on performance metrics. | Continuous optimization based on conversion data can significantly enhance campaign effectiveness.     |\n\n### Creative Insights\n**Key Trends:**\n- Use of sophisticated visuals\n- Emotional storytelling\n- Interactive content formats\n\n### Creative Strategy\n**Description:** Incorporate a mix of image and video formats to engage the audience effectively.\n\n| Recommendation
            | Supporting Insight                                                                                     |\n|--------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------|\n| Utilize high-quality images and engaging video content to showcase the collection and brand story. | Successful fashion campaigns often blend different content formats to maximize engagement.             |\n\n### Creative Content Type\n**Description:** Focus on visually appealing content that highlights the brand\'s unique style and offerings.\n\n| Recommendation
 | Supporting Insight                                                                                     |\n|--------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------|\n| Create lookbooks, behind-the-scenes videos, and influencer collaborations to resonate with the target audience. | Diverse content types can enhance audience engagement and brand loyalty.                               |\n## Campaign Performance Report\n\n**Brand and Product:** Aramis - Personalized Gifts for Loved Ones  \n**Campaign Objective:** Brand Awareness  \nThe budget was calculated based on the total amount allocated for the campaign, which was $1295.73. This budget was used to reach a specific number of impressions and generate clicks.\n\n| Campaign Duration
 |       Budget         | Booked Impressions    | Delivered Impressions | Clicks/Actions  | Value to Money (Conversions) |                ECPM                  |\n|-----------------------------------|----------------------|-----------------------|-----------------------|-----------------|------------------------------|--------------------------------------| \n| Short (30 days)                  |      $1295.73        |     83333
|        86382          |    2855         |        1.88%                 | $15.00                               |\n\n| Creative Snapshot
          | Creative Thumbnail                           |\n|----------------------------------------|----------------------------------------------|\n| Elegant clothing, modern accessories    | ![Ontrip-F3.png](http://127.0.0.1:8000/static/images/0f/41/0f41b14a86b3d582cd7af0ef39387f33.thumbnail.jpg "Aramis - Personalized Gifts for Loved Ones") |\n\n---\n\n**Brand and Product:** Warner Bros - Streaming Service for Sports, News, and Movies  \n**Campaign Objective:** Brand Awareness  \nThe budget was calculated based on the total amount allocated for the campaign, which was $6613.64. This budget was used to reach a significant number of impressions and generate a high volume of clicks.\n\n| Campaign Duration                 |       Budget         | Booked Impressions    | Delivered Impressions | Clicks/Actions  | Value to Money (Conversions) |                ECPM
   |\n|-----------------------------------|----------------------|-----------------------|-----------------------|-----------------|------------------------------|--------------------------------------| \n| Long (60 days)                    |      $6613.64        |     468750            |        165341         |    114010       |        1.85%                 | $40.00                               |\n\n| Creative Snapshot
  | Creative Thumbnail                           |\n|----------------------------------------|----------------------------------------------|\n| Logos, sports scenes, movie clips      | ![OnTrip.mp4](http://127.0.0.1:8000/static/images/e9/c5/e9c5ade14682ace6c978bc56ae168ac9.thumbnail.jpg "Warner Bros - Streaming Service for Sports, News, and Movies") |\n\n---\n\n**Brand and Product:** Qi Tech - Fintech Solutions  \n**Campaign Objective:** Brand Awareness  \nThe budget was calculated based on the total amount allocated for the campaign, which was $1183.935. This budget was used to reach a targeted number of impressions and generate clicks.\n\n| Campaign Duration                 |       Budget         | Booked Impressions    | Delivered Impressions | Clicks/Actions  | Value to Money (Conversions) |                ECPM                  |\n|-----------------------------------|----------------------|-----------------------|-----------------------|-----------------|------------------------------|--------------------------------------| \n| Medium (45 days)                 |      $1183.935       |     75000             |        78929          |    2346         |        1.78%                 | $15.00                               |\n\n| Creative Snapshot                      | Creative Thumbnail
  |\n|----------------------------------------|----------------------------------------------|\n| Golf ball, business context             | ![dispatch5.png](http://127.0.0.1:8000/static/images/48/c9/48c934a1900cc60d9ded69bd95d30e99.thumbnail.jpg "Qi Tech - Fintech Solutions") |\n## Creative Insights\n\n### Creative 1: Aramis\n| Detail                     | Description
           |\n|---------------------------|-------------------------------------------------------------------------------------------------|\n| **Brand**                 | Aramis                                                                                         |\n| **Product**
   | Personalized gifts for loved ones                                                              |\n| **Creative Snapshot**     | A sophisticated and elegant ad showcasing personalized gifts in a minimalistic studio setting, featuring a male model dressed in elegant clothing and modern accessories. |\n| **Creative Thumbnail**     | ![Ontrip-F3.png](http://127.0.0.1:8000/static/images/0f/41/0f41b14a86b3d582cd7af0ef39387f33.thumbnail.jpg "Aramis and Personalized Gifts") |\n| **Brand Elements**        | Elegant clothing, modern accessories, minimalistic design
             |\n| **Seasonal Holiday Elements** |                                                                                             |\n| **Visual Elements**       | Minimalistic studio backdrop, one male model                                                   |\n| **Color Tone**            | Black, white                                                                                    |\n| **Cinematography**        |
                                                                                    |\n| **Audio Elements**        |
                                                                 |\n| **Narrative Structure**   |
                                              |\n\n---\n\n### Creative 2: Warner Bros\n| Detail                     | Description
                                                                  |\n|---------------------------|-------------------------------------------------------------------------------------------------|\n| **Brand**                 | Warner Bros
                            |\n| **Product**               | Streaming service for sports, news, and movies
        |\n| **Creative Snapshot**     | An exciting ad filled with dynamic cuts of sports scenes and movie clips, promoting a streaming service with a vibrant and engaging tone. |\n| **Creative Thumbnail**     | ![OnTrip.mp4](http://127.0.0.1:8000/static/images/e9/c5/e9c5ade14682ace6c978bc56ae168ac9.thumbnail.jpg "Warner Bros and Streaming Service") |\n| **Brand Elements**        | Logos, sports scenes, movie clips
                                      |\n| **Seasonal Holiday Elements** |
                   |\n| **Visual Elements**       | Various locations related to sports and entertainment
 |\n| **Color Tone**            | Blue, pink                                                                                      |\n| **Cinematography**        | Dynamic cuts and engaging visuals                                                                |\n| **Audio Elements**        | Uplifting and energetic music, strong and persuasive voiceover                                   |\n| **Narrative Structure**   | Opening with vibrant clips, middle highlighting features, closing with call to action.          |\n\n---\n\n### Creative 3: Qi Tech\n| Detail                     | Description                                                                                     |\n|---------------------------|-------------------------------------------------------------------------------------------------|\n| **Brand**                 | Qi Tech
                                                     |\n| **Product**               | Fintech solutions
                                   |\n| **Creative Snapshot**     | An inspirational ad set on a golf course, showcasing fintech solutions with a focus on innovation and technology. |\n| **Creative Thumbnail**     | ![dispatch5.png](http://127.0.0.1:8000/static/images/48/c9/48c934a1900cc60d9ded69bd95d30e99.thumbnail.jpg "Qi Tech and Fintech Solutions") |\n| **Brand Elements**        | Golf ball, business context
                                         |\n| **Seasonal Holiday Elements** |
                      |\n| **Visual Elements**       | Golf course setting
    |\n| **Color Tone**            | Green, white                                                                                    |\n| **Cinematography**        |                                                                                                 |\n| **Audio Elements**        |                                                                                                 |\n| **Narrative Structure**   |
                                                                            |\n\n## Performance Summary\n\n### Ad Objective\nThe brand awareness objective was effectively met across all campaigns, with a strong focus on creating emotional connections with the audience through sophisticated visuals and storytelling. Each brand successfully utilized their unique selling propositions to enhance visibility and engagement.\n\n### Call to Action\nClick-through rates varied across campaigns, with Aramis achieving a 3.43% CTR, Warner Bros at 24.36%, and Qi Tech at 3.13%. To improve these rates, it is recommended to refine CTAs to be more action-oriented and personalized, such as \'Discover Your Perfect Gift\' for Aramis and \'Join the Action Now\' for Warner Bros.\n\n### Tone and Mood\nThe festive and sophisticated tone of the campaigns positively impacted audience perception, particularly for Aramis, which resonated well with consumers seeking elegant gifts. Warner Bros\' vibrant tone effectively captured the excitement of sports and entertainment, while Qi Tech\'s inspirational tone aligned well with its innovative fintech solutions.\n\n### Duration Category\nCampaign durations were strategically chosen, with Aramis running for 30 days, Warner Bros for 60 days, and Qi Tech for 45 days. The longer duration of Warner Bros allowed for sustained engagement and visibility, while Aramis\' shorter campaign effectively created urgency around personalized gifts during the holiday season.\n\n### Unique Selling Proposition\nEach brand highlighted its unique selling propositions effectively. Aramis focused on personalized gifts, Warner Bros on a diverse streaming service, and Qi Tech on innovative fintech solutions. Seasonal offerings were emphasized, particularly for Aramis, which aligned well with holiday shopping trends.\n\n### Event Context\nThe holiday season significantly influenced engagement levels, particularly for Aramis, which saw a spike in interest for personalized gifts. Warner Bros capitalized on sports events and movie releases, while Qi Tech\'s campaign was less impacted by seasonal trends but still resonated with its target audience.\n\n### Creative Performance\nCreative elements varied across campaigns. Aramis utilized elegant visuals and minimalistic design, Warner Bros employed dynamic cuts and vibrant scenes, and Qi Tech showcased an inspirational setting. To enhance diversity, it is suggested that future campaigns incorporate more varied content formats, such as user-generated content or interactive elements, to engage audiences further.\n\n### Overall Performance\nOverall, the campaigns were successful in generating high impressions and engagement, with Aramis achieving 86,382 impressions, Warner Bros 165,341 impressions, and Qi Tech 78,929 impressions. However, conversion rates were lower than expected, particularly for Aramis and Qi Tech, indicating a need for improved targeting and follow-up strategies to convert engagement into sales.\n\n\n'}, {'role': 'user', 'content': 'Can you provide some creative insights used in the plan given above'}]"""

In [ ]:
import base64
import requests
from openai import OpenAI

client = OpenAI(api_key="key")
messages = [{
            "role": "user",
            "content": [
                {"type": "text", "text": query},
                # {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_image}"}}
            ]
        }]
messages.append(
    {"role":"system","content":prompt} 
)

response = client.chat.completions.create(
    model="gpt-4o-mini",  # This is typically ignored for local models
    messages=messages,
    max_tokens=500
)

print(response.choices[0].message.content)

In [ ]:
import base64
import requests
from openai import OpenAI

client = OpenAI(api_key="key")
messages = [{
            "role": "user",
            "content": [
                {"type": "text", "text": query},
                # {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_image}"}}
            ]
        }]
messages.append(
    {"role":"system","content":prompt} 
)

response = client.chat.completions.create(
    model="gpt-4o-mini",  # This is typically ignored for local models
    messages=messages,
    max_tokens=500
)

print(response.choices[0].message.content)

### Testing if an image works with spanish or not.

In [ ]:
import base64
import requests
from openai import OpenAI

# Function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# Path to your image
image_path = "GenAI/Dispatch-F1.png"

# Encode the image
base64_image = encode_image(image_path)

# Set up the client to use a local LLM server
client = OpenAI(api_key="api-key")

# Create the prompt
prompt = """Analyze this image and provide the following:
1. A description of what you see in the image.
2. The creative theme or concept behind the image.
3. The design elements used (colors, composition, style, etc.).
Please be detailed in your analysis."""

# Make the API call
response = client.chat.completions.create(
    model="gpt-4o-mini",  # This is typically ignored for local models
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_image}"}}
            ]
        }
    ],
    max_tokens=500
)

# Print the response
print(response.choices[0].message.content)

In [ ]:
import base64
import requests
from openai import OpenAI

# Function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# Path to your image
image_path = "GenAI/Dispatch-F1.png"

# Encode the image
base64_image = encode_image(image_path)

# Set up the client to use a local LLM server
client = OpenAI(api_key="api-key")

# Create the prompt
prompt = """Return output in JSON format. If you are unsure about anything, leave it as an empty string.
        Industry: [Specify the industry]
        Company: [Provide the company name]
        Brand: {brand}
        Brand Type: [Specify brand type (e.g., luxury, service)]
        Brand Mascots: [List any brand mascots]
        Product/Service: [Describe the advertised product or service]
        Product Category: [Specify product category]
        Business Category: [Specify business category]
        Ad Objective: [State the primary goal (like Brand Awareness, Lead Generation, Sales Promotion, Customer Retention)]
        Format: [Specify the ad format(Image, Video, Interactive, Game)]
        Targeting : [What kind of ad targeting methods is used(can be none): {target}]
        Target Market: [Specify the target market for the product/service]
        Target Audience: [Specify demographic or psychographic details]
        Key Message: [Summarize the main takeaway in one sentence]
        Tone and Mood: [Describe the overall atmosphere]
        Start Date: [Extract Start Date from Folder Name: {campaign_folder}. Example 1: SVT_ChurchillDowns_US_Twinspires_01-26-24-01-27-24_JourneyAd extract 01-26-2024. Example 2:Evidens De Beaute_10/04 extract 10-04-2024]
        End Date: [Extract End Date from Folder Name: {campaign_folder} Example 1: SVT_ChurchillDowns_US_Twinspires_01-26-24-01-27-24_JourneyAd extract 01-27-24. Example 2:Evidens De Beaute_10/04 extract NULL]]
        Weekends: [Count number of weekends between the start and end date]
        Holidays: [Specify holidays that occurr around the start and end dates(Leave as Empty string if none found)]
        National Events: [Specify US national events that occurr around the start and end dates(Leave as Empty string if none found)]
        International Events: [Specify relevant international events that occurr around the start and dates(Leave as Empty string if none found)]
        Sport Events: [Specify relevant sports events like Super Bowl that occurr around the start and end dates(Leave as Empty string if none found)]
        Creative Theme: [Describe the creative theme(Humor, Aspirational, Relatable, Cause-Related Marketing, Futuristic, Problem-Solving, Throwback Themes, Minimalism, Trendy, Bold Imagery)]
        Strategy: [Describe the advertising strategy - Emotional Appeal, Lifestyle, Social Responsibility, Innovation and Technology, Nostalgia, Simplicity, Cultural Relevance, Visual Impact]
        Visual Elements:
        Setting: [Describe the primary location(s)]
        Characters: [List main people or animated figures]
        Colors: [Mention the two most dominant colors present in the creative as a string]
        Imagery: [Note significant objects or symbols as a string]
        Creative Description: [Detailed description of the creative]"""

# Make the API call
response = client.chat.completions.create(
    model="gpt-4o-mini",  # This is typically ignored for local models
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_image}"}}
            ]
        }
    ],
    max_tokens=500
)

# Print the response
print(response.choices[0].message.content)

In [ ]:
import base64
import requests
from openai import OpenAI

# Function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# Path to your image
image_path = "GenAI/Dispatch-F1.png"

# Encode the image
base64_image = encode_image(image_path)

# Set up the client to use a local LLM server
client = OpenAI(api_key="api-key")

# Create the prompt
prompt = """Return output in JSON format. If you are unsure about anything, leave it as an empty string.
        Industry: [Specify the industry]
        Company: [Provide the company name]
        Brand: {brand}
        Brand Type: [Specify brand type (e.g., luxury, service)]
        Brand Mascots: [List any brand mascots]
        Product/Service: [Describe the advertised product or service]
        Product Category: [Specify product category]
        Business Category: [Specify business category]
        Ad Objective: [State the primary goal (like Brand Awareness, Lead Generation, Sales Promotion, Customer Retention)]
        Format: [Specify the ad format(Image, Video, Interactive, Game)]
        Targeting : [What kind of ad targeting methods is used(can be none): {target}]
        Target Market: [Specify the target market for the product/service]
        Target Audience: [Specify demographic or psychographic details]
        Key Message: [Summarize the main takeaway in one sentence]
        Tone and Mood: [Describe the overall atmosphere]
        Start Date: [Extract Start Date from Folder Name: {campaign_folder}. Example 1: SVT_ChurchillDowns_US_Twinspires_01-26-24-01-27-24_JourneyAd extract 01-26-2024. Example 2:Evidens De Beaute_10/04 extract 10-04-2024]
        End Date: [Extract End Date from Folder Name: {campaign_folder} Example 1: SVT_ChurchillDowns_US_Twinspires_01-26-24-01-27-24_JourneyAd extract 01-27-24. Example 2:Evidens De Beaute_10/04 extract NULL]]
        Weekends: [Count number of weekends between the start and end date]
        Holidays: [Specify holidays that occurr around the start and end dates(Leave as Empty string if none found)]
        National Events: [Specify US national events that occurr around the start and end dates(Leave as Empty string if none found)]
        International Events: [Specify relevant international events that occurr around the start and dates(Leave as Empty string if none found)]
        Sport Events: [Specify relevant sports events like Super Bowl that occurr around the start and end dates(Leave as Empty string if none found)]
        Creative Theme: [Describe the creative theme(Humor, Aspirational, Relatable, Cause-Related Marketing, Futuristic, Problem-Solving, Throwback Themes, Minimalism, Trendy, Bold Imagery)]
        Strategy: [Describe the advertising strategy - Emotional Appeal, Lifestyle, Social Responsibility, Innovation and Technology, Nostalgia, Simplicity, Cultural Relevance, Visual Impact]
        Visual Elements:
        Setting: [Describe the primary location(s)]
        Characters: [List main people or animated figures]
        Colors: [Mention the two most dominant colors present in the creative as a string]
        Imagery: [Note significant objects or symbols as a string]
        Creative Description: [Detailed description of the creative]"""

# Make the API call
response = client.chat.completions.create(
    model="gpt-4o-mini",  # This is typically ignored for local models
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_image}"}}
            ]
        }
    ],
    max_tokens=500
)

# Print the response
print(response.choices[0].message.content)

In [ ]:
# Set up the client to use a local LLM server
client = OpenAI(api_key="key")

In [ ]:
import json

def classify_with_llm(input, prompt):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": input}
            ],
            # max_tokens=100
        )
        output = response.choices[0].message.content
        print(output)
        return output
    except Exception as e:
        print(f"Error during LLM classification: {e}")
        return None


In [ ]:

def process_text_file(input_file, output_file, prompt):
    """
    Read lines from a text file, classify each line using an LLM,
    and store results in a JSON file.
    :param input_file: Path to the input text file
    :param output_file: Path to the output JSON file
    :param prompt: Classification prompt to be used with the LLM
    """
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            file_content = f.read().strip()
        if not file_content:
            print("Error: Input file is empty.")
            return
        classification = classify_with_llm(file_content, prompt)
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(classification, f, indent=2)
        print(f"Classification result saved to {output_file}")
    
    except FileNotFoundError:
        print(f"Error: Input file {input_file} not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
test_text = """How do I determine the right media channels for my target audience?"""
prompt = "Classify the queries provided into 5 major categories:1. Strategy Development 2. Audience and Targeting, 3. Campaign Planning and Execution, 4. Performance and Optimization, 5. Budget and Compliance and format your output in JSON as follows: [{'Query':'Propose a Marketing Strategy thats alligns with our objective and budget','Classification':''}]"
classify_with_llm(test_text,prompt)

In [ ]:
prompt = "Classify the queries provided into categories and format your output in JSON as follows: [{'Query':'Propose a Marketing Strategy thats alligns with our objective and budget','Classification':''}]"
process_text_file("Creative_Prompts/MediaCreativePrompts.txt","Creative_Prompts/MediaCreativePrompts.json",prompt=prompt)

In [ ]:
prompt = "Classify the queries provided into categories and format your output in JSON as follows: [{'Query':'Propose a Marketing Strategy thats alligns with our objective and budget','Classification':''}]"
process_text_file("Creative_Prompts/MediaPlanningPrompts.txt","Creative_Prompts/MediaPlanningPrompts.json",prompt=prompt)

In [ ]:
import json

# Read the JSON file
with open('Creative_Prompts/MediaPlanningPrompts.json', 'r') as file:
    data = json.load(file)

# Get unique Classifications
unique_classifications = list(set(item['Classification'] for item in data))

print("Unique Classifications:")
for classification in unique_classifications:
    print(f"- {classification}")

In [ ]:
import json

# Read the JSON file
with open('Creative_Prompts/MediaCreativePrompts.json', 'r') as file:
    data = json.load(file)
    
# Get unique Classifications
unique_classifications = list(set(item['Classification'] for item in data))

print("Unique Classifications:")
for classification in unique_classifications:
    print(f"- {classification}")

In [ ]:
process_text_file("Creative_Prompts/MediaCreativePrompts.txt","Creative_Prompts/MediaCreativePrompts.json",prompt=prompt)

In [ ]:
prompt = "Classify the queries provided into 5 major categories:1. Strategy and Planning, 2. Creative Development and Management, 3. Performance and Analytics, 4. Audience and Targeting, 5. Technical and Compliance and format your output in JSON as follows: [{'Query':'Propose a Marketing Strategy thats alligns with our objective and budget','Classification':''}]"
process_text_file("Creative_Prompts/MediaCreativePrompts.txt","Creative_Prompts/MediaCreativePrompts.json",prompt=prompt)

In [ ]:
prompt = "Classify the queries provided into 5 major categories:1. Strategy Development 2. Audience and Targeting, 3. Campaign Planning and Execution, 4. Performance and Optimization, 5. Budget and Compliance and format your output in JSON as follows: [{'Query':'Propose a Marketing Strategy thats alligns with our objective and budget','Classification':''}]"
process_text_file("Creative_Prompts/MediaPlanningPrompts.txt","Creative_Prompts/MediaPlanningPrompts.json",prompt=prompt)

In [ ]:
import json
import csv

with open('Creative_Prompts/MediaCreativePrompts.json', 'r') as file:
    json_data = json.load(file)
with open('Creative_Prompts/MediaCreativePrompts.csv', 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = json_data[0].keys()
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for row in json_data:
        writer.writerow(row)
with open('Creative_Prompts/MediaPlanningPrompts.json', 'r') as file:
    json_data = json.load(file)
with open('Creative_Prompts/MediaPlanningPrompts.csv', 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = json_data[0].keys()
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for row in json_data:
        writer.writerow(row)


# Hashing the Creatives

In [97]:
import cv2
import os
import hashlib
from PIL import Image
import shutil
from typing import Dict
def calculate_md5(file_path: str) -> str:
    hash_md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

def get_file_info(file_path: str) -> Dict[str, str]:
    file_size = os.path.getsize(file_path)
    _, file_extension = os.path.splitext(file_path)
    file_extension = file_extension.lower()
    if file_extension == '.gif':
        nature = "video"
    elif file_extension in ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']:
        nature = "image"
    elif file_extension in ['.mp4', '.avi', '.mkv', '.mov']:
        nature = "video"
    else:
        nature = "unknown"
    return {
        "size": file_size,
        "nature": nature
}
def create_thumbnail(file_path: str, thumbnail_path: str):
    try:
        with Image.open(file_path) as img:
            if img.mode in ('P', 'RGBA'):
                img = img.convert('RGB')
            img.thumbnail((128, 113))
            img.save(thumbnail_path, "JPEG")
    except Exception as e:
        print(f"Error creating thumbnail for {file_path}: {str(e)}")

def extract_video_thumbnail(video_path: str, thumbnail_path: str):
    vidcap = cv2.VideoCapture(video_path)
    success, image = vidcap.read()
    if success:
        image = cv2.resize(image, (128, 113))
        cv2.imwrite(thumbnail_path, image)
    vidcap.release()

In [98]:
def process_and_organize_files(root_dir: str, target_dir: str):
    os.makedirs(target_dir, exist_ok=True)
    images_dir = os.path.join(target_dir, "images")
    os.makedirs(images_dir, exist_ok=True)

    for root, _, files in os.walk(root_dir):
        for filename in files:
            file_path = os.path.join(root, filename)
            file_info = get_file_info(file_path)

            if file_info['nature'] in ['image', 'video']:
                md5_hash = calculate_md5(file_path)
                folder_path = os.path.join(images_dir, md5_hash[:2], md5_hash[2:4])
                os.makedirs(folder_path, exist_ok=True)

                new_file_name = f"{md5_hash}{os.path.splitext(filename)[1]}"
                new_file_path = os.path.join(folder_path, new_file_name)
                shutil.copy(file_path, new_file_path)

                if file_info['nature'] == 'image':
                    thumbnail_path = os.path.join(folder_path, f"{md5_hash}.thumbnail.jpg")
                    create_thumbnail(file_path, thumbnail_path)
                elif file_info['nature'] == 'video':
                    video_thumbnail_path = os.path.join(folder_path, f"{md5_hash}.thumbnail.jpg")
                    extract_video_thumbnail(file_path, video_thumbnail_path)

                print(f"Processed: {filename} -> {new_file_path}")

In [101]:
# C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/GenAI - Uber Creatives"

# Example usage
root_directory = "C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/GenAI - Uber Creatives Old"
target_directory = "C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/static"
process_and_organize_files(root_directory, target_directory)

Processed: Uber 1.png -> C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/static\images\da\1f\da1f44a4c52d61cbb4525744cb3c8635.png
Processed: Uber 2.png -> C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/static\images\6a\0a\6a0a10c0a6ab6d90b08eeaf34dbd3463.png
Processed: Uber 3.png -> C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/static\images\0c\f3\0cf393e4fc763da12d9b59a5d786098f.png
Processed: Uber a.png -> C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/static\images\22\05\2205cb43bade8a9b6854050ce325d5be.png
Processed: Uber b.png -> C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/static\images\bb\87\bb871bbc6ee868448dd156648dad7fcd.png
Processed: Uber c.png -> C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/static\images\09\07\0907c597071eac23c64fdae3f585baa6.png
Processed: HM-Dispatch-F1_v1.png -> C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/static\images\33\69\3369

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/GenAI - Uber Creatives Old\\SVT_Genentech_US_ONSConference_4-23-24-4-29-24_JourneyAd\\Assets for IOPEX\\Tags_2024 _ Polivy _ General _ DLBCL1L _ HCP _ Display_Genentech - Polivy HCP-EMCOutdoor_4-5-24.xlsx'

In [106]:
def process_and_organize_files(root_dir: str, target_dir: str):
    os.makedirs(target_dir, exist_ok=True)
    images_dir = os.path.join(target_dir, "images")
    os.makedirs(images_dir, exist_ok=True)

    successful_count = 0
    total_count = 0
    failed_files = []

    for root, _, files in os.walk(root_dir):
        for filename in files:
            total_count += 1
            file_path = os.path.join(root, filename)
            
            try:
                file_info = get_file_info(file_path)

                if file_info['nature'] in ['image', 'video']:
                    md5_hash = calculate_md5(file_path)
                    folder_path = os.path.join(images_dir, md5_hash[:2], md5_hash[2:4])
                    os.makedirs(folder_path, exist_ok=True)

                    new_file_name = f"{md5_hash}{os.path.splitext(filename)[1]}"
                    new_file_path = os.path.join(folder_path, new_file_name)
                    shutil.copy(file_path, new_file_path)

                    if file_info['nature'] == 'image':
                        thumbnail_path = os.path.join(folder_path, f"{md5_hash}.thumbnail.jpg")
                        create_thumbnail(file_path, thumbnail_path)
                    elif file_info['nature'] == 'video':
                        video_thumbnail_path = os.path.join(folder_path, f"{md5_hash}.thumbnail.jpg")
                        extract_video_thumbnail(file_path, video_thumbnail_path)

                    print(f"Processed: {filename} -> {new_file_path}")
                    successful_count += 1
                else:
                    print(f"Skipped: {filename} (Unsupported file type)")
                    failed_files.append((filename, file_path, "Unsupported file type"))
            except Exception as e:
                print(f"Error processing {filename}: {str(e)}")
                failed_files.append((filename, file_path, str(e)))

    print(f"\nProcessing complete.")
    print(f"Successfully processed: {successful_count} out of {total_count} files.")
    print(f"Success rate: {(successful_count / total_count) * 100:.2f}%")

    return successful_count, total_count, failed_files

In [107]:
root_directory = "C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/GenAI - Uber Creatives Old"
target_directory = "C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/static"
successful, total, failed_files = process_and_organize_files(root_directory, target_directory)

print("\nList of files that failed to process:")
for filename, file_path, error in failed_files:
    print(f"File: {filename}")
    print(f"Path: {file_path}")
    print(f"Error: {error}")
    print("-" * 50)

Processed: Uber 1.png -> C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/static\images\da\1f\da1f44a4c52d61cbb4525744cb3c8635.png
Processed: Uber 2.png -> C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/static\images\6a\0a\6a0a10c0a6ab6d90b08eeaf34dbd3463.png
Processed: Uber 3.png -> C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/static\images\0c\f3\0cf393e4fc763da12d9b59a5d786098f.png
Processed: Uber a.png -> C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/static\images\22\05\2205cb43bade8a9b6854050ce325d5be.png
Processed: Uber b.png -> C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/static\images\bb\87\bb871bbc6ee868448dd156648dad7fcd.png
Processed: Uber c.png -> C:/Users/vishnu.krishnan/Iopex/Creatives/GenAI - Uber Creatives/static\images\09\07\0907c597071eac23c64fdae3f585baa6.png
Skipped: .DS_Store (Unsupported file type)
Processed: HM-Dispatch-F1_v1.png -> C:/Users/vishnu.krishnan/Iopex/Creatives/GenA

### Sending list of Ingested campaigns to brian

In [1]:
import os
import pandas as pd
from openpyxl import Workbook

# Specify the folder path
folder_path = r"GenAI - Uber Creatives/GenAI - Uber Creatives Old"

# Get all files in the folder
files = os.listdir(folder_path)

# Create a pandas DataFrame with the file names
df = pd.DataFrame(files, columns=['Campaign Names'])

# Create an Excel writer object
excel_file = 'campaign_names.xlsx'
with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    # Write the DataFrame to the Excel file
    df.to_excel(writer, sheet_name='Campaign Names', index=False)

print(f"Excel file '{excel_file}' has been created with the list of files.")

Excel file 'campaign_names.xlsx' has been created with the list of files.
